# 04 — Correlation Analysis

Pearson correlation analysis between user attributes.
Headline question (from the resume version of this project):
> *Is **income** correlated with **bio length**?*

We compute the overall correlation plus correlations broken down by cohort to check if the relationship is consistent across sub-populations.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import pearson_correlation, correlation_matrix, cohort_pearson

In [ ]:
df = pd.read_parquet('../data/processed/okcupid_features.parquet')
print(f"Loaded {len(df):,} rows")

## Headline correlation: income ↔ bio length

In [ ]:
result = pearson_correlation(df, 'income', 'bio_length')
print(f"  n (income disclosed):    {result['n']:,}")
print(f"  Pearson r:               {result['r']:.4f}")
print(f"  95% CI:                  [{result['ci_low']:.4f}, {result['ci_high']:.4f}]")
print(f"  p-value:                 {result['p_value']:.2e}")

In [ ]:
sub = df[['income', 'bio_length']].dropna()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(sub['income'], sub['bio_length'].clip(upper=3000), alpha=0.15, s=8, color='#4A90E2')
# Log-binned median trend
sub_q = sub.copy()
sub_q['income_bucket'] = pd.qcut(sub_q['income'], q=10, duplicates='drop')
medians = sub_q.groupby('income_bucket', observed=True).agg(
    income_mid=('income', 'median'),
    bio_median=('bio_length', 'median'),
)
ax.plot(medians['income_mid'], medians['bio_median'], color='red', marker='o', linewidth=2, label='Median per income decile')
ax.set_xlabel('Income (USD)')
ax.set_ylabel('Bio length (characters, clipped at 3000)')
ax.set_title(f'Income vs Bio Length (r = {result["r"]:.3f}, n = {result["n"]:,})')
ax.legend()
plt.tight_layout()
plt.show()

## Correlation matrix across numeric features

In [ ]:
numeric_cols = ['age', 'height', 'income', 'bio_length', 'bio_word_count', 'total_essay_length',
                'essays_written', 'profile_completeness', 'drinks_score', 'smokes_score', 'education_score']
cm = correlation_matrix(df, numeric_cols)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='.2f', center=0, cmap='RdBu_r', vmin=-0.5, vmax=0.5, ax=ax, square=True)
ax.set_title('Pearson correlation matrix')
plt.tight_layout()
plt.show()

## Income vs bio length, broken down by sex

In [ ]:
by_sex = cohort_pearson(df, 'sex', 'income', 'bio_length')
by_sex

## Income vs bio length, broken down by age group

In [ ]:
by_age = cohort_pearson(df, 'age_group', 'income', 'bio_length')
by_age

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
by_age['r'].plot.bar(ax=ax, color='#4A90E2', yerr=[by_age['r'] - by_age['ci_low'], by_age['ci_high'] - by_age['r']], capsize=4)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('Pearson r (income vs bio_length) by age group, with 95% CI')
ax.set_ylabel('Pearson r')
plt.tight_layout()
plt.show()